# Cell 1. Imports

In [0]:
import gzip
import io
import json
import uuid
from datetime import datetime, timezone
from typing import Any, Dict, List

import boto3
import requests

# Cell 2. Widgets (Parameters)

In [0]:
dbutils.widgets.text("base_ccy", "EUR")
dbutils.widgets.dropdown(
    "mode", "daily", ["daily", "hist_90d", "hist_full"]
)  # daily, 90d, or all history(from 1999-01-01)
dbutils.widgets.text("source", "ecb")

# ---- S3 target ----
dbutils.widgets.text("s3_bucket", "enterprise-raw-lakehouse")
dbutils.widgets.text("s3_prefix", "macro/ecb/fx/daily")

# ---- AWS Access Key 输入框 ----
dbutils.widgets.text("aws_access_key", "", "AWS Access Key")
dbutils.widgets.text("aws_secret_key", "", "AWS Secret Key")
dbutils.widgets.text("aws_region", "eu-central-1", "AWS Region")

print("✅ Widgets created. Please configure AWS credentials in the UI.")

# Cell 3. Parsing & Setup

In [0]:
BASE_CCY = dbutils.widgets.get("base_ccy").strip().upper()
MODE = dbutils.widgets.get("mode").strip().lower()
SOURCE = dbutils.widgets.get("source").strip()

BUCKET_NAME = dbutils.widgets.get("s3_bucket").strip()
S3_PREFIX = dbutils.widgets.get("s3_prefix").strip().strip("/")
ACCESS_KEY = dbutils.widgets.get("aws_access_key").strip()
SECRET_KEY = dbutils.widgets.get("aws_secret_key").strip()
REGION = dbutils.widgets.get("aws_region").strip()

URL_DAILY = "https://www.ecb.europa.eu/stats/eurofxref/eurofxref-daily.xml"
URL_90D = "https://www.ecb.europa.eu/stats/eurofxref/eurofxref-hist-90d.xml"
URL_FULL = "https://www.ecb.europa.eu/stats/eurofxref/eurofxref-hist.xml"

if MODE == "daily":
    url = URL_DAILY
elif MODE == "hist_90d":
    url = URL_90D
else:
    url = URL_FULL


print(f"🚀 PRODUCER CONFIG | MODE: {MODE}")
print(f"🔹 Source: {SOURCE}")
print(f"🔹 Base CCY: {BASE_CCY}")
print(f"🔹 Target S3: s3://{BUCKET_NAME}/{S3_PREFIX}/")
print(f"🔹 URL: {url}")

# Cell 4. Initialize S3 Client

In [0]:
if not BUCKET_NAME:
    raise ValueError("❌ s3_bucket is empty")
if not ACCESS_KEY or not SECRET_KEY:
    raise ValueError("❌ Missing AWS creds: aws_access_key / aws_secret_key widgets")
if not REGION:
    raise ValueError("❌ aws_region is empty (e.g. eu-central-1)")

_boto_sess = boto3.session.Session(
    aws_access_key_id=ACCESS_KEY, aws_secret_access_key=SECRET_KEY, region_name=REGION
)

s3 = _boto_sess.client("s3")
print("✅ S3 client ready. bucket:", BUCKET_NAME, "| region:", REGION)

# Cell 5. Fetch Data from ECB API

In [0]:
print(f"📥 Fetching data from: {url}")
r = requests.get(url, timeout=30)
r.raise_for_status()
raw_xml = r.text

print(f"✅ Extracted {len(raw_xml)} bytes of XML data.")

# Cell 6. S3 写入（JSONL.gz）

In [0]:
def _to_iso_z(dt: datetime) -> str:
    return dt.astimezone(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def build_s3_key(mode: str, ingest_dt: datetime) -> str:
    dt_str = ingest_dt.strftime("%Y-%m-%d")
    hhmmss = ingest_dt.strftime("%H%M%S")

    # Path format: s3_prefix/dt=YYYY-MM-DD/ecb_fx_{mode}_{YYYYMMDD_HHMMSS}Z.jsonl.gz
    return f"{S3_PREFIX}/dt={dt_str}/ecb_fx_{mode}_{dt_str}T{hhmmss}Z.jsonl.gz"


def upload_jsonl_gz(bucket: str, key: str, rows: List[Dict[str, Any]]):
    if not rows:
        return

    buf = io.BytesIO()
    with gzip.GzipFile(fileobj=buf, mode="wb") as gz:
        for r in rows:
            line = json.dumps(r, ensure_ascii=False) + "\n"
            gz.write(line.encode("utf-8"))

    s3.put_object(
        Bucket=bucket,
        Key=key,
        Body=buf.getvalue(),
        ContentType="application/x-ndjson",
        ContentEncoding="gzip",
    )
    print(f"💾 Uploaded to s3://{bucket}/{key}")


run_id = str(uuid.uuid4())
ingestion_ts = datetime.now(timezone.utc)

record = {
    "base_ccy": BASE_CCY,
    "raw_json": raw_xml,  # Keep schema compatibility with Databricks bronze table
    "run_id": run_id,
    "ingestion_ts": _to_iso_z(ingestion_ts),
    "source": SOURCE,
    "mode": MODE,
}

key = build_s3_key(MODE, ingestion_ts)
upload_jsonl_gz(BUCKET_NAME, key, [record])

print("[INFO] ECB producer completed successfully.")